In [ ]:
# migrate_hnsw_indexes.py
import os
import sys
import psycopg2

SQLS = [
    "DROP INDEX IF EXISTS idx_article_section_embedding_embedding;",
    """
    CREATE INDEX IF NOT EXISTS idx_article_section_embedding_hnsw
    ON article_section_embedding
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
    """,
    "DROP INDEX IF EXISTS idx_protocol_embedding_embedding;",
    """
    CREATE INDEX IF NOT EXISTS idx_protocol_embedding_hnsw
    ON protocol_embedding
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
    """,
]

def main():
    # 환경변수 우선 (없으면 각자 맞게 DSN 직접 넣어도 됨)
    dsn = os.getenv("POSTGRES_DSN")
    if not dsn:
        host = os.getenv("POSTGRES_HOST", "localhost")
        port = os.getenv("POSTGRES_PORT", "5432")
        db   = os.getenv("POSTGRES_DB", "sknfinaldb")
        user = os.getenv("POSTGRES_USER", "postgres")
        pw   = os.getenv("POSTGRES_PASSWORD", "postgres")
        dsn = f"host={host} port={port} dbname={db} user={user} password={pw}"

    conn = psycopg2.connect(dsn)
    conn.autocommit = False

    try:
        with conn.cursor() as cur:
            # (선택) pgvector 버전 확인 (HNSW는 pgvector v0.5.0+에서 지원)
            cur.execute("SELECT extversion FROM pg_extension WHERE extname='vector';")
            row = cur.fetchone()
            if not row:
                raise RuntimeError("pgvector extension(vector)이 설치되어 있지 않습니다.")
            print(f"[OK] pgvector version: {row[0]}")

            for sql in SQLS:
                cur.execute(sql)
                print("[OK] executed:", " ".join(sql.split())[:120], "...")
        conn.commit()
        print("\n✅ Done: HNSW index migration completed.")
    except Exception as e:
        conn.rollback()
        print("\n❌ Failed. Rolled back.")
        raise
    finally:
        conn.close()

if __name__ == "__main__":
    main()


: 

In [ ]:
# analyze_and_explain.py
import os
import psycopg2

def get_dsn():
    dsn = os.getenv("POSTGRES_DSN")
    if dsn:
        return dsn

    host = os.getenv("POSTGRES_HOST", "localhost")
    port = os.getenv("POSTGRES_PORT", "5432")
    db   = os.getenv("POSTGRES_DB", "postgres")
    user = os.getenv("POSTGRES_USER", "postgres")
    pw   = os.getenv("POSTGRES_PASSWORD", "postgres")
    return f"host={host} port={port} dbname={db} user={user} password={pw}"


def run_analyze_and_explain():
    dsn = get_dsn()
    conn = psycopg2.connect(dsn)
    conn.autocommit = False

    try:
        with conn.cursor() as cur:
            print("▶ ANALYZE article_section_embedding / protocol_embedding ...")
            cur.execute("ANALYZE article_section_embedding;")
            cur.execute("ANALYZE protocol_embedding;")
            print("✅ ANALYZE done.\n")

            # ------------------------------
            # EXPLAIN for article_section_embedding (HNSW)
            # ------------------------------
            print("▶ EXPLAIN article_section_embedding (HNSW index 사용 여부 확인)")
            cur.execute("""
                SET LOCAL hnsw.ef_search = 100;

                EXPLAIN (ANALYZE, BUFFERS)
                SELECT chunk_id
                FROM article_section_embedding
                ORDER BY embedding <=> (
                    SELECT embedding FROM article_section_embedding LIMIT 1
                )
                LIMIT 10;
            """)
            rows = cur.fetchall()
            print("---- article_section_embedding ----")
            for (line,) in rows:
                print(line)
            print()

            # ------------------------------
            # EXPLAIN for protocol_embedding (HNSW)
            # ------------------------------
            print("▶ EXPLAIN protocol_embedding (HNSW index 사용 여부 확인)")
            cur.execute("""
                SET LOCAL hnsw.ef_search = 100;

                EXPLAIN (ANALYZE, BUFFERS)
                SELECT chunking_id
                FROM protocol_embedding
                ORDER BY embedding <=> (
                    SELECT embedding FROM protocol_embedding LIMIT 1
                )
                LIMIT 10;
            """)
            rows = cur.fetchall()
            print("---- protocol_embedding ----")
            for (line,) in rows:
                print(line)

        conn.commit()
        print("\n✅ Done: ANALYZE + EXPLAIN finished.")

    except Exception as e:
        conn.rollback()
        print("\n❌ Failed, rolled back.")
        raise
    finally:
        conn.close()


if __name__ == "__main__":
    run_analyze_and_explain()
